# Weather Data Ingestion

## Objective
Fetch weather data from API and store in Lakehouse.

## API Details
- Source: (your API name)
- Data: temperature, date, hourly metrics

## Process
1. Call API
2. Parse JSON
3. Convert to dataframe
4. Store in Lakehouse

## Output Schema
- Date
- Temperature
- Hour

In [1]:
import requests   # to call API
import pandas as pd  # to handle data

StatementMeta(, 9e4e1553-35b8-4718-9e87-f6f57cbc45db, 3, Finished, Available, Finished, False)

In [2]:
# Step 1: Define API endpoint for location (New York coordinates)
url = "https://api.weather.gov/points/40.7128,-74.0060"

# Step 2: Define headers (required by API)
headers = {
    "User-Agent": "(weather-data-project, vxxxxxxx@gmail.com)"
}

# Step 3: Call API
response = requests.get(url, headers=headers)

# Step 4: Convert response to JSON
data = response.json()

StatementMeta(, 9e4e1553-35b8-4718-9e87-f6f57cbc45db, 4, Finished, Available, Finished, False)

In [3]:
# Step 5: Check structure
print(data.keys())

StatementMeta(, 9e4e1553-35b8-4718-9e87-f6f57cbc45db, 5, Finished, Available, Finished, False)

dict_keys(['@context', 'id', 'type', 'geometry', 'properties'])


In [4]:
# Extract hourly forecast endpoint from metadata
forecast_url = data['properties']['forecastHourly']

print(forecast_url)

StatementMeta(, 9e4e1553-35b8-4718-9e87-f6f57cbc45db, 6, Finished, Available, Finished, False)

https://api.weather.gov/gridpoints/OKX/33,42/forecast/hourly


In [5]:
# Step 5: Fetch hourly weather data using forecast URL

# Call forecast API
forecast_response = requests.get(forecast_url, headers=headers)

# Check if API call was successful
if forecast_response.status_code != 200:
    print("Forecast API call failed:", forecast_response.status_code)

# Convert response to JSON
forecast_data = forecast_response.json()

# Inspect structure of response
print(forecast_data.keys())

StatementMeta(, 9e4e1553-35b8-4718-9e87-f6f57cbc45db, 7, Finished, Available, Finished, False)

dict_keys(['@context', 'type', 'geometry', 'properties'])


In [6]:
import json
import pandas as pd

# Extract Useful Data (periods)
records = forecast_data['properties']['periods']

print("Total records fetched:", len(records))
print("Sample record:", records[0])

# Convert to DataFrame
df_clean = pd.DataFrame(records)

# Add ingestion timestamp
df_clean['ingestion_time'] = pd.Timestamp.utcnow()

df_clean.head()

StatementMeta(, 9e4e1553-35b8-4718-9e87-f6f57cbc45db, 8, Finished, Available, Finished, False)

Total records fetched: 156
Sample record: {'number': 1, 'name': '', 'startTime': '2026-05-03T16:00:00-04:00', 'endTime': '2026-05-03T17:00:00-04:00', 'isDaytime': True, 'temperature': 56, 'temperatureUnit': 'F', 'temperatureTrend': None, 'probabilityOfPrecipitation': {'unitCode': 'wmoUnit:percent', 'value': 0}, 'dewpoint': {'unitCode': 'wmoUnit:degC', 'value': -5.555555555555555}, 'relativeHumidity': {'unitCode': 'wmoUnit:percent', 'value': 26}, 'windSpeed': '22 mph', 'windDirection': 'NW', 'icon': 'https://api.weather.gov/icons/land/day/wind_sct?size=small', 'shortForecast': 'Mostly Sunny', 'detailedForecast': ''}


,number,name,startTime,endTime,isDaytime,temperature,temperatureUnit,temperatureTrend,probabilityOfPrecipitation,dewpoint,relativeHumidity,windSpeed,windDirection,icon,shortForecast,detailedForecast,ingestion_time
0,1,,2026-05-03T16:00:00-04:00,2026-05-03T17:00:00-04:00,True,56,F,None,"{'unitCode': 'wmoUnit:percent', 'value': 0}","{'unitCode': 'wmoUnit:degC', 'value': -5.55555...","{'unitCode': 'wmoUnit:percent', 'value': 26}",22 mph,NW,https://api.weather.gov/icons/land/day/wind_sc...,Mostly Sunny,,2026-05-03 20:15:21.718922+00:00
1,2,,2026-05-03T17:00:00-04:00,2026-05-03T18:00:00-04:00,True,56,F,None,"{'unitCode': 'wmoUnit:percent', 'value': 0}","{'unitCode': 'wmoUnit:degC', 'value': -5}","{'unitCode': 'wmoUnit:percent', 'value': 27}",21 mph,NW,https://api.weather.gov/icons/land/day/wind_sc...,Mostly Sunny,,2026-05-03 20:15:21.718922+00:00
2,3,,2026-05-03T18:00:00-04:00,2026-05-03T19:00:00-04:00,False,57,F,None,"{'unitCode': 'wmoUnit:percent', 'value': 0}","{'unitCode': 'wmoUnit:degC', 'value': -4.44444...","{'unitCode': 'wmoUnit:percent', 'value': 28}",17 mph,W,https://api.weather.gov/icons/land/night/sct?s...,Partly Cloudy,,2026-05-03 20:15:21.718922+00:00
3,4,,2026-05-03T19:00:00-04:00,2026-05-03T20:00:00-04:00,False,58,F,None,"{'unitCode': 'wmoUnit:percent', 'value': 0}","{'unitCode': 'wmoUnit:degC', 'value': -3.33333...","{'unitCode': 'wmoUnit:percent', 'value': 29}",14 mph,W,https://api.weather.gov/icons/land/night/sct?s...,Partly Cloudy,,2026-05-03 20:15:21.718922+00:00
4,5,,2026-05-03T20:00:00-04:00,2026-05-03T21:00:00-04:00,False,58,F,None,"{'unitCode': 'wmoUnit:percent', 'value': 1}","{'unitCode': 'wmoUnit:degC', 'value': -1.66666...","{'unitCode': 'wmoUnit:percent', 'value': 33}",12 mph,W,https://api.weather.gov/icons/land/night/sct?s...,Partly Cloudy,,2026-05-03 20:15:21.718922+00:00


In [7]:
# Convert pandas DataFrame to Spark DataFrame
# Required because Lakehouse uses Spark engine

spark_df = spark.createDataFrame(df_clean)

StatementMeta(, 9e4e1553-35b8-4718-9e87-f6f57cbc45db, 9, Finished, Available, Finished, False)

In [8]:
# Save data to Lakehouse table (Bronze layer)
# Using append mode to simulate incremental ingestion

spark_df.write.mode("append").saveAsTable("weather_raw_bronze")

StatementMeta(, 9e4e1553-35b8-4718-9e87-f6f57cbc45db, 10, Finished, Available, Finished, False)